In [1]:
!pip install pymanopt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.2 MB/s eta 0:00:00


In [2]:
%reset -f

In [3]:
import numpy as np
# import tensorflow as tf
import cupy as cp
# import jax.numpy as jnp
# import jax
# from jax import jit, lax
# from jax import device_put
# from jax.lib import xla_bridge

# import pymanopt
# from pymanopt.manifolds import Oblique
# from pymanopt.optimizers import TrustRegions
#from cupy.dtypes import float8_e4m3fn, float8_e5m2

import time
import gc

In [4]:
# import os

# # 2. Use CUB and cuTENSOR backends (Faster reductions/elementwise)
# os.environ["CUPY_ACCELERATORS"] = "cub,cutensor"

# # 3. Tune the Compiler
# # Tells the GPU compiler to optimize for the A100 architecture specifically
# os.environ["CUDA_CACHE_MAXSIZE"] = "2147483648" # 2GB Cache

# # # TF32 settings
# # # Allow CuPy/CUDA to use Tensor Cores for Float32
# # os.environ["NVIDIA_TF32_OVERRIDE"] = "1"

In [5]:
def reset_memory():
    # Delete all possible GPU arrays
    globals_ = list(globals().keys())
    for var in globals_:
        if var in ['cp', 'jnp', 'reset_memory']: continue
        if isinstance(globals()[var], (cp.ndarray, jnp.ndarray)):
            del globals()[var]

    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

In [6]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))
shift = 50/n

C = cp.random.randn(n,n)/n
C = C + C.T
cp.fill_diagonal(C, C.diagonal() + shift)

# generate_B.seed = 0
# B = generate_B.randn(k,n)
B = cp.random.randn(k,n)
B = B/cp.linalg.norm(B,axis = 0)

# cp_f8 = float8_e5m2
# cp_f8 = float8_e4m3fn

C_32 = cp.array(C,dtype = cp.float32)
B_32 = cp.array(B,dtype = cp.float32)

B_32.dot(C_32)
cp.cuda.Device().synchronize()
# 1000 is good enough do not exceed 1500
K32 = [100,100,100,100,100,100,100,100,100,100]
for i in range(len(K32)):
  cg0 = time.time()
  for t in range(K32[i]):
      B_32 = B_32.dot(C_32)
      B_32 = B_32/cp.linalg.norm(B_32,axis = 0)
  cp.cuda.Device().synchronize()
  cg1 = time.time()
  obj64 = cp.tensordot(B_32,B_32.dot(C))
  print("block time",cg1-cg0)
  print("obj 64",obj64)

block time 2.8347842693328857
obj 64 531.6792445964313
block time 2.5823307037353516
obj 64 532.2191439985053
block time 2.5906190872192383
obj 64 532.2919205721996
block time 2.5932154655456543
obj 64 532.311592292788
block time 2.5954182147979736
obj 64 532.3191043603965
block time 2.610863208770752
obj 64 532.322628110079
block time 2.61084246635437
obj 64 532.3245185073185
block time 2.613678216934204
obj 64 532.3256329793674
block time 2.615354299545288
obj 64 532.3263371604733
block time 2.6171228885650635
obj 64 532.3268059730946


In [7]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))

repeat = 50

f_cg = np.zeros((repeat,10))
t_cg = np.zeros((repeat,10))

shift = 50/n

cp.random.seed(0)
for rep in range(repeat):
  print("----------------REP++",rep,"++---------------------------------------")
  # reset_memory()
  C = cp.random.randn(n,n)/n
  C = C + C.T
  cp.fill_diagonal(C, C.diagonal() + shift)

  # generate_B.seed = 0
  # B = generate_B.randn(k,n)
  B = cp.random.randn(k,n)
  B = B/cp.linalg.norm(B,axis = 0)

  C_32 = cp.array(C,dtype = cp.float32)
  B_32 = cp.array(B,dtype = cp.float32)
  dummy_res = B_32.dot(C_32)
  cp.cuda.Device().synchronize()

  # 1000 is good enough do not exceed 1500
  K32 = [100,100,100,100,100,100,100,100,100,100]
  for i in range(len(K32)):
    cg0 = time.time()
    for t in range(K32[i]):
        B_32 = B_32.dot(C_32)
        B_32 = B_32/cp.linalg.norm(B_32,axis = 0)
    cp.cuda.Device().synchronize()
    cg1 = time.time()

    # obj64 = cp.tensordot(B_32,B_32.dot(C))-shift *n
    obj64 = cp.tensordot(B_32,B_32.dot(C))
    print("block time",cg1-cg0)
    print("obj 64",obj64)
    f_cg[rep,i] = obj64
    t_cg[rep,i] = cg1-cg0

----------------REP++ 0 ++---------------------------------------
block time 2.6159508228302
obj 64 531.7209759424513
block time 2.622875928878784
obj 64 532.2614603337167
block time 2.623892068862915
obj 64 532.33383270591
block time 2.623854398727417
obj 64 532.3532177749305
block time 2.626964807510376
obj 64 532.3605266273772
block time 2.627437114715576
obj 64 532.3638978498628
block time 2.6287126541137695
obj 64 532.3656737499502
block time 2.629495143890381
obj 64 532.3667024327782
block time 2.631355047225952
obj 64 532.3673422347224
block time 2.630932569503784
obj 64 532.3677626732218
----------------REP++ 1 ++---------------------------------------
block time 2.629427671432495
obj 64 531.7590150974877
block time 2.633983612060547
obj 64 532.3146765703739
block time 2.633511781692505
obj 64 532.3884625222119
block time 2.6355233192443848
obj 64 532.407996602441
block time 2.634542226791382
obj 64 532.4153136583124
block time 2.635826349258423
obj 64 532.418690363445
block ti

In [8]:
!mkdir -p /content/drive/MyDrive/result_folder
np.savez("/content/drive/MyDrive/result_folder/results_gpu_float32_final.npz",f_cg = f_cg, t_cg = t_cg)

In [9]:
np.cumsum(t_cg.mean(axis = 0))

array([ 2.64267213,  5.28940494,  7.93598798, 10.58233199, 13.22890984,
       15.87538971, 18.52212515, 21.16892356, 23.81567261, 26.46252522])

In [10]:
f_cg.mean(axis = 0)

array([531.6867906 , 532.23218515, 532.30480641, 532.32425494,
       532.33161723, 532.33503397, 532.33684515, 532.33789927,
       532.33855654, 532.33898887])

In [11]:
print(cp.__version__)

14.0.1


In [12]:
!nvidia-smi

Fri Mar  6 05:16:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   62C    P0            106W /  400W |   34780MiB /  81920MiB |     60%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----